# 24 — Correlation Analysis

Explore how features, sensors, and signals relate to each other across the bearing lifetime.

**Dataset**: XJTU-SY — Bearing 1_1  
**API**: `assay.plot_correlation()` · `study.plot_sensor_lifecycle_correlation()` · `study.plot_cross_correlation()`

In [ ]:
import warnings, logging, sys
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)

WRAPPER_ROOT = Path("..").resolve()
if str(WRAPPER_ROOT) not in sys.path:
    sys.path.insert(0, str(WRAPPER_ROOT))

from isa_phm import ISAWrapper
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

ISA_JSON = Path(r"G:\ISA\Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM-Out.json")
print("Exists:", ISA_JSON.exists())

In [ ]:
wrapper = ISAWrapper(ISA_JSON, strict_validation=False, cache_maxsize=10)

study = wrapper.study("Bearing 1_1")
assay = study.assay(1)   # horizontal accelerometer

print(f"Study  : {study.title}")
print(f"Assay  : {assay.assay_id}  ({assay.run_count} runs)")
print(f"Sensors: {', '.join(a.assay_id for a in study.list_assays())}")

## 1. Feature correlation across runs

`plot_correlation()` computes the Pearson correlation between all lifecycle feature pairs across all runs.  
RMS, std, peak2peak, and max typically cluster together.  
Kurtosis and crest_factor often anti-correlate with RMS in early stages — the classic *kurtosis reversal*.

In [ ]:
fig = assay.plot_correlation(file_type="raw", n_workers=8)
bokeh_show(fig)

## 2. Sensor-to-sensor lifecycle correlation

`study.plot_sensor_lifecycle_correlation()` builds a matrix where each row is a run and each column is a sensor.  
The heatmap shows how the RMS of the horizontal channel correlates with the vertical channel — and whether both degrade simultaneously.

In [ ]:
# Compare RMS degradation across both sensor channels
fig = study.plot_sensor_lifecycle_correlation(
    feature="rms",
    file_type="raw",
    n_workers=8,
)
bokeh_show(fig)

In [ ]:
# Kurtosis — often shows an earlier correlation break between channels
fig = study.plot_sensor_lifecycle_correlation(
    feature="kurtosis",
    file_type="raw",
    n_workers=8,
)
bokeh_show(fig)

## 3. Cross-correlation between two sensors

`plot_cross_correlation()` shows the normalised cross-correlation between two sensors as a function of sample lag (±`max_lag`).  
A peak at lag 0 means the two sensors respond simultaneously.  
A non-zero peak lag indicates one sensor leads the other by that many samples.

In [ ]:
# Healthy run — horizontal vs vertical channel
first_run = assay.list_runs()[0]

fig = study.plot_cross_correlation(
    assay_id_1=1,
    assay_id_2=2,
    run_id=first_run.run_id,
    max_lag=200,
    file_type="raw",
    title=f"Cross-correlation — Healthy (run {first_run.run_number})",
)
bokeh_show(fig)

In [ ]:
# Near-failure run — does the lag change?
last_run = assay.list_runs()[-1]

fig = study.plot_cross_correlation(
    assay_id_1=1,
    assay_id_2=2,
    run_id=last_run.run_id,
    max_lag=200,
    file_type="raw",
    title=f"Cross-correlation — Near Failure (run {last_run.run_number})",
)
bokeh_show(fig)